# Retail Customer Intelligence Workshop — Zadania
## Od danych z Marketplace do AI-powered analizy klientów

**Scenariusz:** Jesteś analitykiem danych w firmie e-commerce specjalizującej się w elektronice użytkowej. Zarząd chce lepiej rozumieć klientów: kto kupuje, co kupuje i jak przewidzieć przyszłe zachowania zakupowe. Dane pobierasz z **Databricks Marketplace** — gotowy, zweryfikowany dataset symulowanych klientów B2B.

**Co zbudujesz:**
1. Eksploracja danych z Marketplace (SQL)
2. Zaawansowana analityka (CTE, Window Functions)
3. AI Functions — LLM w zapytaniach SQL
4. PySpark — Cleaning, Join, Feature Engineering → `gold_customer_360`
5. Delta Lake — zapis i wersjonowanie
6. Tool Calling — UC Function jako narzędzie LLM
7. Machine Learning z MLflow (klasyfikacja loyalty_segment)
8. Model Registry w Unity Catalog
9. Batch Inference + AutoML
10. Prognoza przychodów + Dashboard + Genie Space

**Dane:** Katalog `databricks_simulated_retail_customer_data` z Marketplace — tabele: `customers` (28 813 klientów), `sales_orders` (4 074 zamówień), `sales` (360 produktów)

**Instrukcje:**
- Każda sekcja zawiera opis zadania, kroki do wykonania i hinty
- Komórki kodu są puste — to Twoja przestrzeń do pisania
- Jeśli utkniesz, zajrzyj do notebooka źródłowego: *Retail Forecasting Workshop od danych do AI*

## Sekcja 1: Dane z Marketplace i eksploracja
**Funkcjonalności:** Databricks Marketplace, Unity Catalog (3-poziomowa hierarchia: catalog.schema.table), SQL w notebooku, wizualizacje inline

### Skąd pochodzą dane?
Dane do tego warsztatu pochodzą z **Databricks Marketplace** — wbudowanego katalogu gotowych datasetów:
1. Otwórz **Marketplace** w panelu bocznym Databricks
2. Wyszukaj `Databricks Simulated Retail Customer Data`
3. Kliknij **Get instant access** — dataset pojawia się jako nowy katalog w Unity Catalog

Po pobraniu mamy katalog `databricks_simulated_retail_customer_data` ze schematem `v01` i trzema tabelami:
- **customers** — dane klientów B2B (customer_id, name, state, loyalty_segment, units_purchased, lat/lon)
- **sales_orders** — zamówienia (order_number, customer_id, order_datetime, number_of_line_items, ordered_products jako JSON)
- **sales** — produkty (product_id, product_name, category, price)

### Czym jest eksploracja danych?
Eksploracja (EDA) to pierwszy krok każdego projektu analitycznego. Zanim budujemy modele, musimy **zrozumieć dane**: ile ich mamy, jaki mają zakres, czy są kompletne.

### Sekcja 1b: Zanim klikniesz „Get instant access” — przegląd warunków licencji

Dataset z Marketplace to **produkt danych dostawcy**, nie nasze dane. Zanim trafi do pipeline’u, Compliance Officer chce wiedzieć **na jakich warunkach** go używamy.

W listingu Marketplace sprawdź i **zanotuj** (to zapis do audytu):

| Co sprawdzić | Gdzie w listingu | Dlaczego |
| --- | --- | --- |
| **Dostawca** i typ produktu | Nagłówek, sekcja *Provider* | Kto odpowiada za jakość danych |
| **Licencja / Terms of use** | Sekcja *Terms* lub *License* | Jakie użycie jest dozwolone (produkcja? redystrybucja?) |
| **Polityka prywatności** | Link w warunkach | Czy dane zawierają PII? Jakie regulacje? |
| **Częstotliwość aktualizacji** | Opis lub metadata | Czy dane są jednorazowe czy odświeżane? |
| **Scope danych** | Opis datasetu | Ile wierszy, jaki okres, jakie tabele |

> **Ćwiczenie:** Otwórz Marketplace, znajdź listing i wypełnij tabelę powyżej.

In [0]:
%%sql
-- ZADANIE 1.1: Zbadaj metadane katalogu z Marketplace
--
-- Katalog z Marketplace to katalog typu Delta Sharing —
-- Unity Catalog przechowuje nazwę dostawcy i share'a.
--
-- Napisz zapytanie, które pokaże metadane katalogu:
--   DESCRIBE CATALOG EXTENDED databricks_simulated_retail_customer_data
--
-- Zwróć uwagę na pola: Catalog Type, Provider Name, Share Name, Owner, Created.
--
-- Następnie wyświetl listę tabel udostępnionych w tym katalogu:
--   SHOW TABLES IN databricks_simulated_retail_customer_data.v01

In [0]:
%%sql
-- ZADANIE 1.2: Ile mamy danych? Jaki zakres?
--
-- Napisz zapytanie UNION ALL, które pokaże dla każdej z 3 tabel:
--   - Nazwę tabeli (stała tekstowa)
--   - Liczbę wierszy (COUNT(*))
--   - Liczbę unikalnych identyfikatorów (COUNT(DISTINCT ...))
--
-- Tabele:
--   databricks_simulated_retail_customer_data.v01.customers   → customer_id
--   databricks_simulated_retail_customer_data.v01.sales_orders → customer_id
--   databricks_simulated_retail_customer_data.v01.sales        → product_id
--
-- Hint: SELECT 'customers' AS tabela, COUNT(*) AS wiersze, COUNT(DISTINCT customer_id) AS unikalne_id
--       FROM ... UNION ALL SELECT 'sales_orders', ...

In [0]:
%%sql
-- ZADANIE 1.3: Rozkład klientów per segment lojalności i stan
--
-- loyalty_segment: 0 = nowy, 1 = okazjonalny, 2 = regularny, 3 = VIP
--
-- Napisz zapytanie GROUP BY, które pokaże:
--   - loyalty_segment, state
--   - COUNT(*) AS liczba_klientow
--   - ROUND(AVG(units_purchased), 1) AS avg_units_purchased
--   - ROUND(AVG(lat), 4) AS avg_lat
-- Z tabeli: databricks_simulated_retail_customer_data.v01.customers
-- Posortuj po loyalty_segment, liczba_klientow DESC
--
-- Po uruchomieniu: kliknij "+" przy wyniku → wybierz wykres słupkowy
-- (X = state, Y = liczba_klientow, Color = loyalty_segment)

## Sekcja 2: Zaawansowana analityka SQL
**Funkcjonalności:** CTE (Common Table Expressions), Window Functions, LAG, RANK, analiza kohortowa

Teraz przechodzimy do bardziej zaawansowanych zapytań — łączymy tabele klientów z zamówieniami, obliczamy trendy tygodniowe i budujemy ranking klientów.

### Kluczowe koncepcje:
- **CTE (Common Table Expression)** — `WITH ... AS (...)` — "tymczasowa tabela" w zapytaniu. Zamiast gnieźddżić podzapytania, rozbijasz logikę na czytelne kroki.
- **Window Functions** — funkcje operujące na "oknie" wierszy powiązanych z bieżącym wierszem:
  - `LAG(col, n)` — wartość z N wierszy wstecz (np. sprzedaż z poprzedniego tygodnia)
  - `RANK() OVER (PARTITION BY ... ORDER BY ...)` — ranking w ramach grupy
- **TRY_DIVIDE** — bezpieczne dzielenie (zwraca NULL zamiast błędu przy dzieleniu przez 0)

In [0]:
%%sql
-- ZADANIE 2.1: Trend tygodniowy zamówień z growth rate (Week-over-Week)
--
-- Odpowiadamy na pytanie: "Jak zmienia się liczba zamówień z tygodnia na tydzień?"
--
-- Napisz zapytanie z CTE i Window Functions:
--
-- CTE 1 (weekly_orders): Agreguj zamówienia tygodniowo
--   - DATE_TRUNC('week', from_unixtime(order_datetime)) AS week
--   - COUNT(*) AS total_orders
--   - SUM(number_of_line_items) AS total_items
--   - COUNT(DISTINCT customer_id) AS unique_customers
-- Źródło: databricks_simulated_retail_customer_data.v01.sales_orders
--
-- CTE 2 (with_growth): Dodaj kolumny z Window Functions
--   - LAG(total_orders, 1) OVER (ORDER BY week) AS prev_week_orders
--   - ROUND(TRY_DIVIDE(total_orders - prev, prev) * 100, 1) AS wow_growth_pct
--
-- Zapytanie końcowe: SELECT * FROM with_growth ORDER BY week
--
-- Hint: TRY_DIVIDE zamiast zwykłego dzielenia — nie wyrzuci błędu gdy prev = 0
-- Po uruchomieniu: dodaj wykres liniowy (X = week, Y = total_orders)
--
-- TO JEST PRZYKŁAD — jeśli chcesz, zrób własną analizę!
-- Ważne jest użycie CTE + LAG/TRY_DIVIDE, nie konkretna metryka. Alternatywy:
--   • Trend miesięczny zamiast tygodniowego (DATE_TRUNC('month', ...))
--   • Growth rate wartości zamówień zamiast liczby
--   • Trend per segment lojalnosci (PARTITION BY loyalty_segment)
--   • Analiza dnia tygodnia: które dni mają najwięcej zamówień?

In [0]:
%%sql
-- ZADANIE 2.2: Top 3 klienci w każdym segmencie lojalności
--
-- Ranking pozwala odpowiedzieć: "Którzy klienci są najlepsi w każdym segmencie?"
-- Używamy RANK() OVER (PARTITION BY ... ORDER BY ...)
--
-- Napisz zapytanie z CTE i podzapytaniem:
--
-- CTE (customer_orders): JOIN customers + sales_orders
--   - c.customer_id, c.customer_name, c.state, c.loyalty_segment
--   - COUNT(o.order_number) AS total_orders
--   - SUM(o.number_of_line_items) AS total_items
--   GROUP BY wszystkie pola klienta
--
-- JOIN: INNER JOIN na customer_id
-- Tabele:
--   databricks_simulated_retail_customer_data.v01.customers c
--   databricks_simulated_retail_customer_data.v01.sales_orders o
--
-- Podzapytanie: Dodaj RANK() OVER (PARTITION BY loyalty_segment ORDER BY total_orders DESC)
-- Filtruj: WHERE rank_in_segment <= 3
-- Sortuj: ORDER BY loyalty_segment, rank_in_segment
--
-- TO JEST PRZYKŁAD — ważne jest użycie RANK() + PARTITION BY. Alternatywy:
--   • Top 3 klienci per STAN (PARTITION BY state) zamiast per segment
--   • Ranking wg wartości monetary zamiast liczby zamówień
--   • Top 5 zamiast top 3 (WHERE rank_in_segment <= 5)
--   • Dodaj DENSE_RANK i porównaj różnice z RANK

## Sekcja 3: AI Functions — sztuczna inteligencja w SQL
**Funkcjonalności:** `ai_query()`, `ai_classify()` — LLM bezpośrednio w zapytaniach SQL

To jedna z najciekawszych możliwości Databricks — wywołujesz model językowy (LLM) jako zwykłą funkcję SQL! Nie potrzebujesz Pythona, API keys, ani infrastruktury.

### Dwie główne funkcje:
- **`ai_query(endpoint, prompt)`** — wysyła tekst do LLM i zwraca odpowiedź. Używasz jak każdej innej funkcji SQL (SUM, AVG, CONCAT...), ale dostaje się do LLM.
- **`ai_classify(tekst, ARRAY('kat1', 'kat2', ...))`** — AI automatycznie przypisuje etykietę z podanej listy kategorii.

### Ważna optymalizacja:
AI Functions są kosztowne (każde wywołanie = request do LLM). Dlatego **najpierw agregujemy** dane, a dopiero potem wywołujemy AI na małym zbiorze wyników. Nigdy nie wywołuj AI na 28 000 wierszach!

In [0]:
%%sql
-- ZADANIE 3.1: AI generuje biznesowy insight o segmentach klientów
--
-- Cel: Dla każdego segmentu lojalności LLM analizuje statystyki
-- i generuje krótką rekomendację biznesową (1–2 zdania).
--
-- Napisz zapytanie z CTE:
--
-- CTE (segment_stats): Agreguj dane per loyalty_segment
--   - COUNT(*) AS customers
--   - ROUND(AVG(units_purchased), 1) AS avg_purchases
--   - ROUND(AVG(units_purchased) * COUNT(*), 0) AS estimated_total_units
-- Źródło: databricks_simulated_retail_customer_data.v01.customers
--
-- SELECT końcowy: loyalty_segment, customers, avg_purchases, plus:
--   ai_query(
--     'databricks-meta-llama-3-3-70b-instruct',
--     CONCAT('Jesteś analitykiem retail. Segment ', CAST(loyalty_segment AS STRING),
--            ' ma ', CAST(customers AS STRING), ' klientów, ',
--            'średnia zakupów: ', CAST(avg_purchases AS STRING), '. ',
--            'Napisz 1–2 zdania rekomendacji biznesowej po polsku.')
--   ) AS ai_recommendation
--
-- Hint: ai_query zwraca STRING — można go użyć w SELECT jak każdą inną kolumnę
--
-- TO JEST PRZYKŁAD — ważne jest użycie ai_query z CTE. Alternatywy:
--   • Rekomendacje per STAN (top 5 stanów) zamiast per segment
--   • Poproś LLM o analizę ryzyka churnu zamiast rekomendacji
--   • Generuj 3-punktową strategię marketingową per segment
--   • Zmień język promptu na angielski i porównaj wyniki

In [0]:
%%sql
-- ZADANIE 3.2: AI klasyfikuje typ klienta na podstawie zachowań
--
-- Cel: AI automatycznie przypisuje etykietę strategii obsługi
-- na podstawie danych klienta (bez ręcznego definiowania progów).
--
-- Napisz zapytanie z CTE:
--
-- CTE (top_customers): Pobierz top 10 klientów z NY i CA
--   - c.customer_name, c.state, c.units_purchased, c.loyalty_segment
--   - COUNT(o.order_number) AS total_orders
--   LEFT JOIN sales_orders o ON customer_id
--   WHERE c.state IN ('NY', 'CA')
--   GROUP BY ..., ORDER BY units_purchased DESC, LIMIT 10
--
-- SELECT końcowy: Dodaj kolumnę ai_classify:
--   ai_classify(
--     CONCAT('Customer: ', customer_name, ', zakupy: ', units_purchased,
--            ' szt., zamówienia: ', total_orders, ', segment: ', loyalty_segment),
--     ARRAY('VIP Premium', 'Regularny Aktywny', 'Rozwijający się', 'Nowy/Do Aktywacji')
--   ) AS ai_customer_type
--
-- Hint: ai_classify zwraca jeden z elementów ARRAY — najlepiej pasujący
-- Hint: LIMIT 10 w CTE — optymalizacja kosztów AI
--
-- TO JEST PRZYKŁAD — ważne jest użycie ai_classify z ARRAY kategorii. Alternatywy:
--   • Własne kategorie: ARRAY('Ryzyko churnu', 'Stabilny', 'Rosnący', 'Nowy')
--   • Klasyfikuj klientów z innych stanów (TX, FL, IL)
--   • Klasyfikuj na podstawie RFM zamiast zakupów
--   • Dodaj więcej kategorii (5-6) i sprawdź czy AI radzi sobie z rozróżnieniem

## Sekcja 4: PySpark — Cleaning, Join i Feature Engineering
**Funkcjonalności:** DataFrame API, JSON parsing, JOIN, Window Functions, wielojęzyczność notebooka

Przechodzimy na Python! W tej sekcji:
1. **Łączymy tabele** — customers + sales_orders (JOIN)
2. **Parsujemy JSON** — wyciągamy dane o produktach i promocjach z kolumny `ordered_products` (STRING z JSON array)
3. **Budujemy cechy klienta** — RFM (Recency, Frequency, Monetary), cechy geograficzne
4. **Przygotowujemy dane do ML** — predykcja `loyalty_segment`

### Czym jest RFM?
- **Recency** — ile dni minęło od ostatniego zamówienia (im mniej, tym lepiej)
- **Frequency** — ile razy klient kupował (więcej = bardziej lojalny)
- **Monetary** — ile łącznie wydał (wartość klienta)

> **Wielojęzyczność:** Zwróć uwagę, że SQL używaliśmy wyżej, a teraz przechodzimy na Python w tym samym notebooku.

In [0]:
# ZADANIE 4.1: Feature Engineering — zbuduj tabelę customer_360
#
# Cel: Z tabel customers i sales_orders stworzysz jedną tabelę Gold
# z pełnym profilem każdego klienta (cechy RFM + statystyki zamówień).
#
# Krok 1: Importy i wczytanie danych
#   from pyspark.sql import functions as F
#   from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType
#   catalog = "databricks_simulated_retail_customer_data"
#   customers = spark.table(f"{catalog}.v01.customers")
#   orders = spark.table(f"{catalog}.v01.sales_orders")
#
# Krok 2: Parsowanie JSON z ordered_products
#   Kolumna ordered_products to string z JSON array — wyciągnij łączną wartość:
#   schema = ArrayType(StructType([StructField("id", StringType()), ...]))
#   orders_parsed = orders.withColumn("products", from_json("ordered_products", schema))
#   orders_value = orders_parsed.withColumn("order_value",
#       F.aggregate("products", F.lit(0.0), lambda acc, x: acc + x["qty"] * x["price"]))
#
# Krok 3: Agregacja per klient (RFM + statystyki)
#   order_stats = orders_value.groupBy("customer_id").agg(
#       F.count("*").alias("num_orders"),
#       F.sum("order_value").alias("monetary"),
#       F.avg("order_value").alias("avg_item_value"),
#       F.min(F.from_unixtime("order_datetime")).alias("first_order_date"),
#       F.max(F.from_unixtime("order_datetime")).alias("last_order_date"),
#       F.sum("number_of_line_items").alias("frequency"),
#       F.sum(F.when(F.col("promo_info").isNotNull(), 1).otherwise(0)).alias("promo_orders")
#   )
#
# Krok 4: JOIN z customers (LEFT JOIN — nie każdy klient ma zamówienia)
#   gold_df = customers.join(order_stats, "customer_id", "left")
#
# Krok 5: Dodaj cechy obliczane:
#   - recency_days = datediff(current_date(), last_order_date)
#   - has_orders = CASE WHEN num_orders > 0 THEN true ELSE false
#   - promo_ratio = promo_orders / num_orders
#   Wypełnij nulle (fillna) dla klientów bez zamówień
#
# Krok 6: display(gold_df.limit(10))
#
# Hint: from_json wymaga zdefiniowania schematu JSON
# Hint: F.aggregate to wbudowana funkcja PySpark do redukcji tablic
# Hint: fillna({"num_orders": 0, "monetary": 0.0, ...}) dla nullów

## Sekcja 5: Delta Lake — zapis i wersjonowanie
**Funkcjonalności:** Delta Lake, saveAsTable, Time Travel, DESCRIBE HISTORY

Zapisujemy przetworzoną tabelę `gold_customer_360` jako tabelę Delta w Unity Catalog. To będzie nasza **główna tabela** — używana w ML, dashboardach, i w następnych warsztatach (guardrails, RAG, agent).

Delta Lake daje nam:
- **ACID transactions** — niezawodność zapisu (albo się uda cały, albo wcale)
- **Time Travel** — dostęp do poprzednich wersji danych
- **Schema enforcement** — ochrona przed błędnymi danymi

> **Ważne:** Ta tabela zawiera dane PII klientów (customer_name, tax_id, adresy) — w Warsztacie 2 zabezpieczymy ją guardrails.

In [0]:
# ZADANIE 5.1: Zapisz DataFrame gold_df jako tabelę Delta w Unity Catalog
#
# Krok 1: Zdefiniuj nazwę tabeli Gold:
#   GOLD_TABLE = "<YOUR_CATALOG>.<YOUR_SCHEMA>.gold_customer_360"
#
# Krok 2: Zapisz dane:
#   gold_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(GOLD_TABLE)
#
# Krok 3: Zweryfikuj zapis:
#   print(f"Tabela zapisana: {GOLD_TABLE}")
#   print(f"  Wiersze: {spark.table(GOLD_TABLE).count():,}")
#   print(f"  Kolumny: {len(spark.table(GOLD_TABLE).columns)}")
#
# Hint: mode("overwrite") nadpisze istniejącą tabelę (jeśli już istnieje)

In [0]:
%%sql
-- ZADANIE 5.2: Time Travel — historia zmian tabeli
--
-- Każdy zapis tworzy nową wersję — możemy wrócić do dowolnej!
--
-- a) Pokaż historię zmian tabeli:
--    DESCRIBE HISTORY <YOUR_CATALOG>.<YOUR_SCHEMA>.gold_customer_360
--
-- b) Sprawdź schemat tabeli Gold (zwróć uwagę na kolumny PII!):
--    DESCRIBE TABLE <YOUR_CATALOG>.<YOUR_SCHEMA>.gold_customer_360
--
-- c) (Opcjonalnie) Odczytaj dane z poprzedniej wersji:
--    SELECT * FROM <YOUR_CATALOG>.<YOUR_SCHEMA>.gold_customer_360 VERSION AS OF 0 LIMIT 5

In [0]:
%%sql
-- ZADANIE 5.2b: Schemat tabeli Gold — sprawdź kolumny PII
--
-- Zwróć uwagę, które kolumny zawierają dane osobowe (PII)!
-- W Warsztacie 2 zabezpieczymy je guardrails.
--
-- a) Schemat tabeli:
--    DESCRIBE TABLE <YOUR_CATALOG>.<YOUR_SCHEMA>.gold_customer_360
--
-- b) Odczytaj dane z wersji 0 (pierwsza wersja po zapisie):
--    SELECT * FROM <YOUR_CATALOG>.<YOUR_SCHEMA>.gold_customer_360 VERSION AS OF 0 LIMIT 5
--
-- Kolumny PII: customer_name, tax_id, lat, lon
-- Kolumny bezpieczne: loyalty_segment, monetary, recency_days, frequency, num_orders

## Sekcja 5b: Tool Calling — UC Function jako narzędzie LLM
**Funkcjonalności:** UC Functions, Foundation Model API, tool calling (function calling)

W sekcji 3 widzieliśmy `ai_query()` — LLM odpowiada na pytanie **w SQL**. Ale co jeśli chcemy, żeby **LLM sam zdecydował jaką funkcję wywołać**? To jest **tool calling**:

```
Użytkownik: „Jaki jest łączny przychód od VIPów w NY?”
   ↓
LLM decyduje: "Muszę wywołać get_revenue_summary(segment=3, state='NY')"
   ↓
UC Function wykonuje SQL na tabeli Gold i zwraca wynik
   ↓
LLM formatuje odpowiedź dla użytkownika
```

To fundament **agentów AI** — w Warsztacie 4 zbudujemy pełnego agenta z wieloma narzędziami.

In [0]:
%%sql
-- ZADANIE 5b.1: Utwórz UC Function zwracającą podsumowanie przychodu
--
-- Ta funkcja będzie narzędziem LLM — agent wywoła ją automatycznie.
--
-- CREATE OR REPLACE FUNCTION <YOUR_CATALOG>.<YOUR_SCHEMA>.get_revenue_summary(
--   segment BIGINT COMMENT 'Loyalty segment ID: 0–3. Pass -1 for all.',
--   state_filter STRING COMMENT 'US state code (e.g. NY). Pass ALL for all states.'
-- )
-- RETURNS STRING
-- COMMENT 'Returns total revenue, customer count and avg revenue per segment/state.'
-- RETURN SELECT CONCAT_WS('\n',
--   CONCAT('Segment: ', COALESCE(CAST(... AS STRING), 'ALL')),
--   CONCAT('State: ', ...),
--   CONCAT('Customers: ', COUNT(*)),
--   CONCAT('Total Revenue: $', ROUND(SUM(monetary), 2)),
--   CONCAT('Avg Revenue: $', ROUND(AVG(monetary), 2))
-- )
-- FROM <YOUR_CATALOG>.<YOUR_SCHEMA>.gold_customer_360
-- WHERE (segment = -1 OR loyalty_segment = segment)
--   AND (state_filter = 'ALL' OR state = state_filter)
--
-- Po utworzeniu przetestuj: SELECT <YOUR_CATALOG>.<YOUR_SCHEMA>.get_revenue_summary(3, 'NY')
--
-- TO JEST PRZYKŁAD — ważne jest stworzenie UC Function z COMMENT i parametrami. Alternatywy:
--   • Funkcja `get_churn_risk(segment)` — zwraca avg recency_days i % bez zamówień
--   • Funkcja `get_state_summary(state)` — liczba klientów, segmenty, avg monetary per stan
--   • Funkcja `compare_segments(seg_a, seg_b)` — porównanie dwóch segmentów w jednym wywołaniu
--   • Dodaj trzeci parametr np. `min_monetary` filtrujący klientów poniżej progu

In [0]:
# ZADANIE 5b.2: Tool calling — LLM sam wywołuje UC Function
#
# Krok 1: Setup klienta OpenAI (Foundation Model API)
#   from openai import OpenAI
#   from databricks.sdk import WorkspaceClient
#   import json
#   w = WorkspaceClient()
#   client = OpenAI(
#       api_key=w.config.authenticate()["Authorization"].split(" ", 1)[1],
#       base_url=f"{w.config.host}/serving-endpoints"
#   )
#
# Krok 2: Zdefiniuj narzędzie (tool) opisujące UC Function:
#   tools = [{
#     "type": "function",
#     "function": {
#       "name": "get_revenue_summary",
#       "description": "Returns total revenue per segment and state",
#       "parameters": {
#         "type": "object",
#         "properties": {
#           "segment": {"type": "integer", "description": "Loyalty segment 0–3, -1 for all"},
#           "state_filter": {"type": "string", "description": "US state code or ALL"}
#         }, "required": ["segment", "state_filter"]
#       }
#     }
#   }]
#
# Krok 3: Wyślij pytanie z narzędziami:
#   response = client.chat.completions.create(
#     model="databricks-meta-llama-3-3-70b-instruct",
#     messages=[{"role": "user", "content": "Jaki jest przychód od VIPów w Nowym Jorku?"}],
#     tools=tools,
#     tool_choice="auto"
#   )
#
# Krok 4: Jeśli LLM wywołał narzędzie — wykonaj funkcję i zwróć wynik:
#   tool_call = response.choices[0].message.tool_calls[0]
#   args = json.loads(tool_call.function.arguments)
#   result = spark.sql(
#       "SELECT <YOUR_CATALOG>.<YOUR_SCHEMA>.get_revenue_summary(:seg, :state)",
#       args={"seg": args["segment"], "state": args["state_filter"]}
#   ).collect()[0][0]
#   print(f"LLM chce wywołać: {tool_call.function.name}({args})")
#   print(f"Wynik: {result}")
#
# TO JEST PRZYKŁAD — ważne jest zrozumienie pętli tool calling. Alternatywy:
#   • Zmień pytanie: „Jaka jest średnia wartość klienta w Kalifornii?” i sprawdź args
#   • Zadaj pytanie wymagające dwóch wywołań: „Porównaj przychód VIP w NY vs CA”
#   • Zadaj pytanie które NIE pasuje do narzędzia i sprawdź co LLM zrobi
#   • Dodaj drugie narzędzie (np. get_customer_profile) i testuj routing

## Sekcja 6: Machine Learning z MLflow
**Funkcjonalności:** scikit-learn, MLflow autologging, tracking eksperymentów, porównanie runów

MLflow to open-source platforma do zarządzania cyklem życia modeli ML. W Databricks jest w pełni zintegrowana:
- **Autologging** — `mlflow.sklearn.autolog()` automatycznie zapisuje parametry, metryki, model
- **Tracking** — każdy trening ("run") jest zapisany, możesz porównać setki eksperymentów w UI
- **Model Registry** — najlepszy model rejestrujesz jako wersję w Unity Catalog

### Nasz problem ML:
**Klasyfikacja loyalty_segment** — na podstawie cech klienta (RFM, geografia, zamówienia) przewidujemy, do którego segmentu lojalności należy klient (0–3).

### Metryki:
- **Accuracy** — % poprawnych predykcji (prosta, ale może być myląca przy niezbalansowanych klasach)
- **F1-score (weighted)** — harmoniczna średnia precyzji i recall, ważona licznością klas

> **Plan:** Trenujemy 2 modele (Gradient Boosting i Random Forest), porównujemy metryki, najlepszy rejestrujemy w UC.

In [0]:
# ZADANIE 6.1: Przygotowanie danych do modelu klasyfikacji
#
# Krok 1: Importy
#   import mlflow, mlflow.sklearn
#   from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
#   from sklearn.model_selection import train_test_split
#   from sklearn.metrics import accuracy_score, f1_score, classification_report
#   import pandas as pd, numpy as np
#
# Krok 2: Wczytaj dane Gold i przekonwertuj na pandas
#   GOLD_TABLE = "<YOUR_CATALOG>.<YOUR_SCHEMA>.gold_customer_360"
#   feature_cols = ["units_purchased", "recency_days", "frequency", "num_orders",
#                   "monetary", "avg_item_value", "promo_orders", "promo_ratio",
#                   "has_orders", "lat", "lon"]  # 11 cech — spójne z głównym notebookiem WS1
#   target_col = "loyalty_segment"
#
#   pdf = (spark.table(GOLD_TABLE)
#       .select(feature_cols + [target_col])
#       .dropna()
#       .toPandas())
#
# Krok 3: Podział na X (features) i y (target)
#   X = pdf[feature_cols]
#   y = pdf[target_col]
#
# Krok 4: train_test_split
#   X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
#   print(f"Train: {len(X_train)}, Test: {len(X_test)}, Klasy: {sorted(y.unique())}")

In [0]:
# ZADANIE 6.2: Trening modelu Gradient Boosting Classifier z MLflow
#
# Krok 1: Włącz autologging
#   mlflow.sklearn.autolog()
#
# Krok 2: Trenuj model w kontekście MLflow run:
#   with mlflow.start_run(run_name="GBClassifier_loyalty"):
#       model_gb = GradientBoostingClassifier(
#           n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42
#       )
#       model_gb.fit(X_train, y_train)
#       preds_gb = model_gb.predict(X_test)
#
#       acc = accuracy_score(y_test, preds_gb)
#       f1 = f1_score(y_test, preds_gb, average='weighted')
#       print(f"Gradient Boosting — Accuracy: {acc:.4f}, F1: {f1:.4f}")
#       print(classification_report(y_test, preds_gb))
#
# Hint: Po uruchomieniu sprawdź MLflow Experiment w panelu bocznym
# Hint: autolog() automatycznie loguje parametry, metryki i model

In [0]:
# ZADANIE 6.3: Trening modelu Random Forest Classifier (do porównania)
#
# Ten sam schemat co 6.2, ale inny algorytm:
#   with mlflow.start_run(run_name="RFClassifier_loyalty"):
#       model_rf = RandomForestClassifier(
#           n_estimators=200, max_depth=10, random_state=42
#       )
#       model_rf.fit(X_train, y_train)
#       preds_rf = model_rf.predict(X_test)
#
#       acc = accuracy_score(y_test, preds_rf)
#       f1 = f1_score(y_test, preds_rf, average='weighted')
#       print(f"Random Forest — Accuracy: {acc:.4f}, F1: {f1:.4f}")
#       print(classification_report(y_test, preds_rf))
#
# Hint: Po uruchomieniu obu modeli sprawdź MLflow Experiment —
#   zobaczysz oba runy obok siebie z metrykami

## Sekcja 7: Model Registry w Unity Catalog
**Funkcjonalność:** Rejestracja modelu, wersjonowanie, governance modeli

Model Registry to **centralny rejestr modeli ML** — zamiast trzymać model jako plik pickle, rejestrujemy go w Unity Catalog:
- **Wersjonowanie** — każda nowa wersja modelu ma swój numer (v1, v2...)
- **Governance** — uprawnienia, audyt, liniaż danych
- **Współdzielenie** — inny zespół może załadować Twój model jedną linią kodu

In [0]:
# ZADANIE 7.1: Rejestracja najlepszego modelu w Unity Catalog
#
# Krok 1: Pobierz ostatni run MLflow:
#   best_run = mlflow.last_active_run()
#
# Krok 2: Zarejestruj model:
#   model_name = "<YOUR_CATALOG>.<YOUR_SCHEMA>.loyalty_classifier"
#   registered_model = mlflow.register_model(
#       model_uri=f"runs:/{best_run.info.run_id}/model",
#       name=model_name
#   )
#
# Krok 3: Wypisz informacje:
#   print(f"Model: {model_name}")
#   print(f"Wersja: {registered_model.version}")
#   print(f"Run ID: {best_run.info.run_id}")
#
# Krok 4: Ustaw alias @champion (WS2-WS4 załadują model przez ten alias):
#   from mlflow import MlflowClient
#   MlflowClient().set_registered_model_alias(
#       name=model_name, alias="champion", version=registered_model.version
#   )
#   print(f"Alias @champion → v{registered_model.version}")
#
# Hint: mlflow.last_active_run() zwraca ostatni run (Random Forest z 6.3)

## Sekcja 8: Predykcja na nowych danych (Batch Inference)
**Funkcjonalność:** Model jako Spark UDF, skalowalna inferencja

Batch Inference to zastosowanie modelu na **dużym zbiorze danych naraz**:
1. Ładujemy model z Unity Catalog: `mlflow.pyfunc.load_model("models:/nazwa/latest")`
2. Model działa jak **zwykła funkcja** — podajesz DataFrame, dostajesz predykcje
3. Dzięki Spark może przetworzyć miliony wierszy równolegle

To typowy scenariusz: co noc generujesz predykcje segmentu lojalności dla nowych klientów.

In [0]:
# ZADANIE 8.1: Załaduj model z UC i wykonaj predykcję
#
# Krok 1: Załaduj model:
#   import os
#   os.environ["MLFLOW_OPENAI_RETRIES"] = "0"
#   model_name = "<YOUR_CATALOG>.<YOUR_SCHEMA>.loyalty_classifier"
#   loaded_model = mlflow.pyfunc.load_model(f"models:/{model_name}/latest")
#
# Krok 2: Przygotuj dane testowe (pandas):
#   test_sample = X_test.head(20)
#
# Krok 3: Wykonaj predykcję:
#   predictions = loaded_model.predict(test_sample)
#
# Krok 4: Porównaj z rzeczywistymi wartościami:
#   comparison = pd.DataFrame({
#       "Actual": y_test.head(20).values,
#       "Predicted": predictions
#   })
#   comparison["Correct"] = comparison["Actual"] == comparison["Predicted"]
#   display(comparison)
#   print(f"Trafność: {comparison['Correct'].mean():.1%}")

## Sekcja 8b: AutoML — automatyczny dobór modelu
**Funkcjonalność:** Databricks AutoML, automatyczne porównanie modeli, feature importance

AutoML to **automatyczny dobieralnik modeli**. Zamiast ręcznie testować różne algorytmy, AutoML robi to za Ciebie:
1. Testuje wiele algorytmów (XGBoost, LightGBM, RandomForest...)
2. Optymalizuje hiperparametry
3. Generuje **gotowy notebook** z najlepszym modelem
4. Loguje wszystko do MLflow

> **Wymaga klastra z ML Runtime** (np. 16.x ML). Na Serverless compute AutoML nie jest dostępny.
> Przed uruchomieniem tej sekcji przybij się na klaster z ML Runtime.

In [0]:
# ZADANIE 8b.1: Uruchom AutoML (klasyfikacja) na danych Gold
# UWAGA: Wymaga klastra z ML Runtime!
#
# Krok 1: Import AutoML:
#   try:
#       from databricks import automl
#   except ImportError:
#       print("AutoML wymaga klastra z ML Runtime")
#       automl = None
#
# Krok 2: Przygotuj dane (te same features co w Sekcji 6):
#   automl_df = spark.table("<YOUR_CATALOG>.<YOUR_SCHEMA>.gold_customer_360").select(
#       "units_purchased", "recency_days", "frequency", "num_orders",
#       "monetary", "avg_item_value", "promo_orders", "promo_ratio",
#       "has_orders", "lat", "lon",
#       "loyalty_segment"
#   ).dropna()
#
# Krok 3: Uruchom AutoML:
#   summary = automl.classify(
#       dataset=automl_df,
#       target_col="loyalty_segment",
#       primary_metric="f1",
#       timeout_minutes=5,
#       max_trials=10
#   )
#
# Krok 4: Wypisz najlepszy model:
#   print(summary.best_trial.model_description)
#   print(summary.best_trial.metrics)

In [0]:
# ZADANIE 8b.2: Załaduj najlepszy model AutoML i porównaj z ręcznym
#
# Krok 1: Załaduj model AutoML:
#   best_model_uri = f"runs:/{summary.best_trial.mlflow_run_id}/model"
#   automl_model = mlflow.pyfunc.load_model(best_model_uri)
#
# Krok 2: Predykcja na X_test (te same dane co w Sekcji 6):
#   automl_predictions = automl_model.predict(X_test)
#
# Krok 3: Oblicz metryki i porównaj:
#   from sklearn.metrics import accuracy_score, f1_score
#   acc_automl = accuracy_score(y_test, automl_predictions)
#   f1_automl = f1_score(y_test, automl_predictions, average='weighted')
#   print(f"AutoML   — Accuracy: {acc_automl:.4f}, F1: {f1_automl:.4f}")
#   print(f"Ręczny GB — Accuracy: {acc_gb:.4f}, F1: {f1_gb:.4f}")
#   print(f"Ręczny RF — Accuracy: {acc_rf:.4f}, F1: {f1_rf:.4f}")
#
# Hint: Który model wygrał? Sprawdź F1 — wyższy = lepszy
# Hint: AutoML też loguje wyniki do MLflow Experiment — sprawdź panel boczny

## Sekcja 9: Prognoza przychodów (Time Series Forecasting)
**Funkcjonalność:** RandomForestRegressor, cechy czasowe, prognoza szeregów czasowych

Prognoza (forecasting) to **przewidywanie przyszłych wartości** na podstawie historycznych danych. Tutaj prognozujemy dzienny przychód na 30 dni do przodu.

### Jak to robimy?
Używamy **Random Forest** trenowanego na **cechach czasowych** wyciągniętych z daty zamówień:
- Dzień tygodnia, miesiąc, tydzień roku, weekend (0/1), trend

### Przedziały ufności
Random Forest składa się z wielu drzew decyzyjnych. Rozrzut ich predykcji to naturalna miara **niepewności**. Używamy percentyli (2.5% i 97.5%) jako granic prognozy.

In [0]:
# ZADANIE 9.1: Prognoza dziennego przychodu (30 dni)
#
# Krok 1: Przygotuj dane czasowe z zamówień:
#   from sklearn.ensemble import RandomForestRegressor
#   import pandas as pd, numpy as np
#
#   orders = spark.table("databricks_simulated_retail_customer_data.v01.sales_orders")
#   daily_revenue = (orders
#       .withColumn("date", F.to_date(F.from_unixtime("order_datetime")))
#       .groupBy("date").agg(F.sum("number_of_line_items").alias("revenue"))
#       .toPandas().sort_values("date"))
#
# Krok 2: Funkcja make_time_features(dates):
#   def make_time_features(dates):
#       df = pd.DataFrame({"date": dates})
#       df["day_of_week"] = df["date"].dt.dayofweek
#       df["month"] = df["date"].dt.month
#       df["week_of_year"] = df["date"].dt.isocalendar().week.astype(int)
#       df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)
#       df["trend"] = range(len(df))
#       return df.drop(columns=["date"])
#
# Krok 3: Trenuj model:
#   X = make_time_features(daily_revenue["date"])
#   y = daily_revenue["revenue"]
#   model = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42)
#   model.fit(X, y)
#
# Krok 4: Prognoza 30 dni:
#   future_dates = pd.date_range(daily_revenue["date"].max() + pd.Timedelta(days=1), periods=30)
#   X_future = make_time_features(future_dates)
#   X_future["trend"] = range(len(X), len(X) + 30)
#   pred = model.predict(X_future)
#
# Krok 5: Przedziały ufności:
#   tree_preds = np.array([t.predict(X_future) for t in model.estimators_])
#   lower = np.percentile(tree_preds, 2.5, axis=0)
#   upper = np.percentile(tree_preds, 97.5, axis=0)
#
# Krok 6: display() wyników jako Spark DataFrame
#
# TO JEST PRZYKŁAD — ważne jest użycie cech czasowych + RF Regressor. Alternatywy:
#   • Prognozuj liczbę zamówień zamiast przychodu
#   • Prognozuj 14 dni zamiast 30 (i porównaj przedziały ufności)
#   • Dodaj cechę „miesiąc” lub „quartał” do modelu i sprawdź feature importance
#   • Spróbuj GradientBoostingRegressor zamiast RF i porównaj R²

## Sekcja 10: Udostępnianie wyników — Dashboard + Genie Space
**Funkcjonalność:** Databricks SDK, Lakeview Dashboard API, Genie Spaces API

Najlepszy model i analiza są **bezwartościowe**, jeśli wyniki nie dotrą do osób podejmujących decyzje.

### Co tworzymy?
- **AI/BI Dashboard** — interaktywny dashboard z wykresami, KPI, tabelami. Tworzymy go **programowo** z kodu.
- **Genie Space** — "chatbot" nad danymi. Użytkownik biznesowy pisze pytanie po polsku, a Genie generuje SQL i zwraca odpowiedź. Nie musi znać SQL!

In [0]:
# ZADANIE 10.1: Tworzenie AI/BI Dashboard programowo (Lakeview API)
#
# Krok 1: Setup SDK
#   import json, requests
#   from databricks.sdk import WorkspaceClient
#   w = WorkspaceClient()
#   host = w.config.host
#   headers = w.config.authenticate()
#
# Krok 2: Zdefiniuj datasety SQL (lista dictów):
#   datasets = [
#     {"name": "kpi_metrics", "displayName": "KPI",
#      "query": "SELECT COUNT(*) as total_customers, ... FROM <YOUR_CATALOG>.<YOUR_SCHEMA>.gold_customer_360"},
#     {"name": "segment_distribution", ...},
#     {"name": "revenue_by_state", ...}
#   ]
#
# Krok 3: Zdefiniuj widgety (counter, bar, pie):
#   Każdy widget = {"name": ..., "queries": [...], "spec": {"widgetType": ..., "encodings": ...}}
#
# Krok 4: Layout strony i serializacja:
#   serialized = json.dumps({"datasets": datasets, "pages": [{"name": "overview", ...}]})
#
# Krok 5: POST do Lakeview API:
#   resp = requests.post(f"{host}/api/2.0/lakeview/dashboards",
#       headers=headers,
#       json={"display_name": "Customer Intelligence Dashboard", "serialized_dashboard": serialized})
#
# Hint: Spec widgetów: version=2 dla counter, version=3 dla bar/pie
# Hint: Sprawdź notebook źródłowy po szczegóły formatów
#
# TO JEST PRZYKŁAD — ważna jest umiejętność tworzenia dashboardu z kodu. Alternatywy:
#   • Dodaj widget z prognozą z Sekcji 9 (line chart z prediction + confidence)
#   • Dodaj mapę geograficzną (avg monetary per state)
#   • Zmień KPI: zamiast total_customers pokaż churn_rate lub avg_monetary
#   • Dodaj filtr interaktywny per segment lojalnosci

In [0]:
# ZADANIE 10.2: Tworzenie Genie Space (REST API)
#
# Genie Space to "chatbot" nad danymi — użytkownik biznesowy pisze pytanie,
# a Genie automatycznie generuje SQL i zwraca odpowiedź.
#
# Krok 1: Pobierz user_email i warehouse_id:
#   user_email = spark.sql("SELECT current_user()").collect()[0][0]
#   wh_response = requests.get(f"{host}/api/2.0/sql/warehouses", headers=headers)
#   warehouse_id = ... (wybierz dostępny warehouse)
#
# Krok 2: Zdefiniuj serialized_space (JSON):
#   - sample_questions: przykładowe pytania po polsku
#     ("Ilu mamy klientów VIP?", "Jaki jest średni przychód per segment?")
#   - data_sources.tables: <YOUR_CATALOG>.<YOUR_SCHEMA>.gold_customer_360
#   - instructions: "Odpowiadaj po polsku."
#
# Krok 3: POST do Genie API:
#   response = requests.post(
#       f"{host}/api/2.0/genie/spaces",
#       headers=headers,
#       json={
#           "title": "Customer Intelligence Assistant",
#           "description": "Asystent do analizy klientów B2B",
#           "serialized_space": serialized_space,
#           "warehouse_id": warehouse_id,
#           "parent_path": f"/Workspace/Users/{user_email}"
#       })
#
# Hint: import uuid; uuid.uuid4().hex dla unikalnych ID
#
# TO JEST PRZYKŁAD — ważne jest stworzenie Genie Space z kodu. Alternatywy:
#   • Napisz własne sample_questions dostosowane do Twojej analizy
#   • Dodaj instrukcje: „Odpowiadaj krótko, max 2 zdania” lub „Dodawaj wykresy”
#   • Podepnij dodatkowe tabele (sales_orders, sales) obok gold_customer_360
#   • Spróbuj zadać pytanie po angielsku i porównać jakość odpowiedzi

## Podsumowanie — co zbudowaliśmy w tym warsztacie

| # | Sekcja | Funkcjonalność Databricks | Język |
|---|---|---|---|
| 1 | Marketplace i eksploracja | Unity Catalog, SQL, Marketplace, wizualizacje | SQL |
| 2 | Zaawansowana analityka | CTE, Window Functions, LAG, RANK | SQL |
| 3 | AI Functions | ai_query(), ai_classify() — LLM w SQL | SQL |
| 4 | Feature Engineering | PySpark DataFrame API, JSON parsing, JOIN, RFM | Python |
| 5 | Delta Lake | saveAsTable, Time Travel, DESCRIBE HISTORY | Python + SQL |
| 5b | Tool Calling | UC Functions, Foundation Model API | SQL + Python |
| 6 | Machine Learning | scikit-learn, MLflow autologging, klasyfikacja | Python |
| 7 | Model Registry | Rejestracja w Unity Catalog | Python |
| 8 | Batch Inference | Model jako Spark UDF | Python |
| 8b | AutoML | Databricks AutoML, automatyczny dobór modelu | Python (ML Runtime) |
| 9 | Forecasting | Random Forest, cechy czasowe, prognoza | Python |
| 10 | Dashboard + Genie | Lakeview API, Genie Spaces API | Python (SDK) |

**Następny krok:** Warsztat 2 — Guardrails, Monitoring i Ewaluacja